# 03 – Baseline Models
Logistic Regression and Decision Tree, binary and multi-class.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.config import RANDOM_STATE         

TRAIN_PATH = "../data/processed/train.parquet"
TEST_PATH  = "../data/processed/test.parquet"

In [2]:
train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)
print(train.shape, test.shape)

(341633, 64) (85409, 64)


In [3]:
LABEL_COLS = ["label_multi", "label_binary"]

X_train = train.drop(columns=LABEL_COLS)
X_test  = test.drop(columns=LABEL_COLS)

y_train_bin   = train["label_binary"]
y_test_bin    = test["label_binary"]
y_train_multi = train["label_multi"]
y_test_multi  = test["label_multi"]

print(X_train.shape, X_test.shape)

(341633, 62) (85409, 62)


In [4]:
print(y_train_bin.value_counts())
print(y_train_multi.value_counts())

label_binary
0    186996
1    154637
Name: count, dtype: int64
label_multi
Benign    186996
Dos       101205
DDoS       40215
Bot         7165
Prob        6052
Name: count, dtype: int64


In [5]:
import time
from sklearn.metrics import f1_score, accuracy_score, classification_report

In [6]:
def evaluate(model, name, task, X_tr, y_tr, X_te, y_te):
    # 1) train and measure time
    t0 = time.perf_counter()
    model.fit(X_tr,y_tr)
    train_time = time.perf_counter() - t0

    # 2) predict and measure time
    t0 = time.perf_counter()
    y_pred = model.predict(X_te)
    predict_time = time.perf_counter() - t0
    predict_ms = predict_time / len(X_te) * 1000 #converts seconds to ms

    # 3) show the detailed report
    print(f"=== {name} | {task} ===")
    print(classification_report(y_te,y_pred,digits=4))

    # 4) return one row for results.csv
    return{
        "model":name,
        "task":task,
        "imbalance":"none",
        "macro_f1":f1_score(y_te,y_pred,average="macro"),
        "accuracy":accuracy_score(y_te,y_pred),
        "train_time_s":round(train_time,2),
        "predict_ms_per_flow":round(predict_ms,5),
        "notes":"",

    }


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [ ]:
result = [] # we collect one dictionary per model here

lr_bin = Pipeline([
    ("scaler",StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

row = evaluate(lr_bin, "LogisticRegression","binary",
               X_train,y_train_bin,X_test,y_test_bin)
result.append(row)
row


=== LogisticRegession | binary ===
              precision    recall  f1-score   support

           0     0.9711    0.8623    0.9135     46749
           1     0.8534    0.9690    0.9075     38660

    accuracy                         0.9106     85409
   macro avg     0.9123    0.9157    0.9105     85409
weighted avg     0.9178    0.9106    0.9108     85409



{'model': 'LogisticRegession',
 'task': 'binary',
 'imbalance': 'none',
 'macro_f1': 0.9105184731460395,
 'accuracy': 0.9106183189125268,
 'train_time_s': 50.37,
 'predict_ms_per_flow': 0.00031,
 'notes': ''}